<a href="https://colab.research.google.com/github/vrnc-juga/Modelos-CNN/blob/MobileNetV2/Modelo_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ─── CELDA 1: Imports ───────────────────────────────────────────
import numpy as np
import gc
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import TensorBoard
from sklearn.model_selection import train_test_split
from google.colab import drive

In [ ]:


# ─── CELDA 2: Montar Drive y cargar datos ───────────────────────
drive.mount('/content/drive')

b = np.load('/content/drive/MyDrive/imagenes_ent225.npy', allow_pickle=True)
n = len(b)
print(f"Total imágenes: {n}")



Mounted at /content/drive
Total imágenes: 7177


In [ ]:
# ─── CELDA 3: Guardar en disco en lugar de RAM ──────────────────
X = np.memmap('/content/X.dat', dtype='float32', mode='w+', shape=(n, 224, 224, 3))
y = np.memmap('/content/y.dat', dtype='int32',   mode='w+', shape=(n,))

for i, (img, label) in enumerate(b):
    X[i] = img
    y[i] = label
    if i % 500 == 0:
        print(f"Progreso: {i}/{n}")

del b
gc.collect()
print("✅ Datos cargados")


Progreso: 0/7177
Progreso: 500/7177
Progreso: 1000/7177
Progreso: 1500/7177
Progreso: 2000/7177
Progreso: 2500/7177
Progreso: 3000/7177
Progreso: 3500/7177
Progreso: 4000/7177
Progreso: 4500/7177
Progreso: 5000/7177
Progreso: 5500/7177
Progreso: 6000/7177
Progreso: 6500/7177
Progreso: 7000/7177
✅ Datos cargados


In [ ]:

# ─── CELDA 4: Verificar normalización ───────────────────────────
print(f"Valor mínimo: {X[0].min():.3f}")
print(f"Valor máximo: {X[0].max():.3f}")
# Si imprime valores entre 0-255, descomenta la siguiente línea:
# X = X / 255.0



Valor mínimo: 0.000
Valor máximo: 1.000


In [ ]:
# ─── CELDA 5: Split y one-hot encoding ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_clases = len(np.unique(y))
print(f"Número de clases: {num_clases}")

y_train = to_categorical(y_train, num_clases)
y_test  = to_categorical(y_test,  num_clases)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")



Número de clases: 7
Train: (5741, 224, 224, 3) | Test: (1436, 224, 224, 3)


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

IMG_SIZE = 224
NUM_CLASSES = 7

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# ─── CELDA 7: Entrenar ───────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=16
)



Epoch 1/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 62s 112ms/step - accuracy: 0.8434 - loss: 0.4892 - val_accuracy: 0.9248 - val_loss: 0.2217
Epoch 2/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.9324 - loss: 0.2141 - val_accuracy: 0.9352 - val_loss: 0.1861
Epoch 3/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.9462 - loss: 0.1714 - val_accuracy: 0.9408 - val_loss: 0.1672
Epoch 4/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.9493 - loss: 0.1485 - val_accuracy: 0.9436 - val_loss: 0.1624
Epoch 5/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.9551 - loss: 0.1341 - val_accuracy: 0.9422 - val_loss: 0.1645
Epoch 6/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.9537 - loss: 0.1306 - val_accuracy: 0.9401 - val_loss: 0.1657
Epoch 7/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.9615 - loss: 0.1154 - val_accuracy: 0.9373 - val_loss: 0.1622
Epoch 8/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.9650 - loss: 0.1017 - val_ac

In [ ]:
# ─── CELDA 8: Evaluar ────────────────────────────────────────────
loss, acc = model.evaluate(X_test, y_test)
print(f"Loss: {loss:.4f}")
print(f"Accuracy: {acc:.4f}")

45/45 ━━━━━━━━━━━━━━━━━━━━ 26s 314ms/step - accuracy: 0.9519 - loss: 0.1438
Loss: 0.1438
Accuracy: 0.9519
